# Punto 1: Clasificación para Focalización de Programas Sociales
Taller 1 Consultoría Económica con IA Responsable

David Rodríguez y Juan Rueda

**Contexto**
- Cliente: BID
- Lugar: Costa Rica
- Programa: Subsidios de asistencia social
- Problema: el modelo vigente presenta problemas de precisión, ya que algunos hogares vulnerables son excluidos
del programa (falsos negativos) y otros que no lo necesitan reciben el subsidio (falsos positivos)


# 1. Importación base de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)

In [2]:
url = "https://raw.githubusercontent.com/darc-17/Sandbox_HE2_DavidRodriguez/refs/heads/main/Taller%201/Punto%201.%20Clasificacion%20para%20Focalizaci%C3%B3n%20de%20Programas%20Sociales/train.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()

(7187, 143)


,Id,v2a1,hacdor,rooms,hacapo,v14a,refrig,v18q,v18q1,r4h1,r4h2,r4h3,r4m1,r4m2,r4m3,r4t1,r4t2,r4t3,tamhog,tamviv,escolari,rez_esc,hhsize,paredblolad,paredzocalo,...,tipovivi4,tipovivi5,computer,television,mobilephone,qmobilephone,lugar1,lugar2,lugar3,lugar4,lugar5,lugar6,area1,area2,age,SQBescolari,SQBage,SQBhogar_total,SQBedjefe,SQBhogar_nin,SQBovercrowding,SQBdependency,SQBmeaned,agesq,Target
0,ID_279628684,190000.0,0,3,0,1,1,0,NaN,0,1,1,0,0,0,0,1,1,1,1,10,NaN,1,1,0,...,0,0,0,0,1,1,1,0,0,0,0,0,1,0,43,100,1849,1,100,0,1.000000,0.0,100.0,1849,4
1,ID_f29eb3ddd,135000.0,0,4,0,1,1,1,1.0,0,1,1,0,0,0,0,1,1,1,1,12,NaN,1,0,0,...,0,0,0,0,1,1,1,0,0,0,0,0,1,0,67,144,4489,1,144,0,1.000000,64.0,144.0,4489,4
2,ID_d671db89c,180000.0,0,5,0,1,1,1,1.0,0,2,2,1,1,2,1,3,4,4,4,9,1.0,4,1,0,...,0,0,0,0,1,3,1,0,0,0,0,0,1,0,17,81,289,16,121,4,1.777778,1.0,121.0,289,4
3,ID_d56d6f5f5,180000.0,0,5,0,1,1,1,1.0,0,2,2,1,1,2,1,3,4,4,4,11,NaN,4,1,0,...,0,0,0,0,1,3,1,0,0,0,0,0,1,0,37,121,1369,16,121,4,1.777778,1.0,121.0,1369,4
4,ID_ec05b1a7b,180000.0,0,5,0,1,1,1,1.0,0,2,2,1,1,2,1,3,4,4,4,11,NaN,4,1,0,...,0,0,0,0,1,3,1,0,0,0,0,0,1,0,38,121,1444,16,121,4,1.777778,1.0,121.0,1444,4


# 2. Limpieza
1. Eliminación de variables redundantes
2. Missing values
3. Dummies con variable cerca a cero
4. Outliers en variables continuas clave
5. Estandarización variable edjefe y edjefa
6. Variables adicionales a eliminar

In [3]:
# 1. Eliminación de variables redundantes
drop_cols = [
    # SQB — cuadrados redundantes
    "SQBescolari", "SQBage", "agesq", "SQBhogar_total",
    "SQBedjefe", "SQBhogar_nin", "SQBovercrowding",
    "SQBdependency", "SQBmeaned",
    # Tamaño hogar — duplicados exactos o correlación >0.95
    "tamhog", "hhsize", "tamviv",
    # Totales = suma de partes
    "r4h3", "r4m3", "r4t3",
    # Género — redundante (se mantiene female)
    "male",
]
drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

print(f"\n[LIMPIEZA] Eliminadas {len(drop_cols)} columnas redundantes.")
print(f"  Shape tras limpieza sección 1: {df.shape}")


[LIMPIEZA] Eliminadas 16 columnas redundantes.
  Shape tras limpieza sección 1: (7187, 127)


In [4]:
# 2. Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({
    "missing_n": missing,
    "missing_%": missing_pct
}).query("missing_n > 0").sort_values("missing_%", ascending=False)

print(f"\nVariables con missing ({len(miss_df)} en total):")
print(miss_df.to_string())

# Análisis específico de v2a1 (renta)
print("\n[v2a1] Renta mensual — cruce con tipo de vivienda:")
rent_cross = df.groupby("tipovivi3")["v2a1"].agg(
    missing=lambda x: x.isnull().sum(),
    no_missing=lambda x: x.notnull().sum()
)
print(rent_cross)

# Análisis de v18q1 (tablets) vs v18q (tiene tablet)
print("\n[v18q1] Tablets — missing cuando v18q=0?")
has_tablet = df[df["v18q"] == 1]["v18q1"].isnull().sum()
no_tablet  = df[df["v18q"] == 0]["v18q1"].isnull().sum()
print(f"  Missing en v18q1 cuando v18q=1 (tiene tablet): {has_tablet}")
print(f"  Missing en v18q1 cuando v18q=0 (no tiene):     {no_tablet}")

# Análisis de rez_esc (retraso escolar)
print("\n[rez_esc] Retraso escolar — missing por rango de edad:")
df["_age_group"] = pd.cut(df["age"], bins=[0,5,18,65,120],
                           labels=["0-5","6-18","19-65","65+"])
rez_miss = df.groupby("_age_group", observed=True)["rez_esc"].apply(
    lambda x: f"{x.isnull().sum()} missing ({x.isnull().mean()*100:.1f}%)"
)
print(rez_miss.to_string())
df.drop(columns=["_age_group"], inplace=True)



Variables con missing (4 en total):
          missing_n  missing_%
rez_esc        5953      82.83
v18q1          5525      76.87
v2a1           5214      72.55
meaneduc          5       0.07

[v2a1] Renta mensual — cruce con tipo de vivienda:
           missing  no_missing
tipovivi3                     
0             5214         712
1                0        1261

[v18q1] Tablets — missing cuando v18q=0?
  Missing en v18q1 cuando v18q=1 (tiene tablet): 0
  Missing en v18q1 cuando v18q=0 (no tiene):     5525

[rez_esc] Retraso escolar — missing por rango de edad:
_age_group
0-5       477 missing (100.0%)
6-18       257 missing (17.2%)
19-65    4498 missing (100.0%)
65+       651 missing (100.0%)


C:\Users\David\AppData\Local\Temp\ipykernel_9668\3120528291.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_age_group"] = pd.cut(df["age"], bins=[0,5,18,65,120],


In [5]:
## Depuración

# v18q1: missing = no tiene tablet → imputar 0
df["v18q1"] = df["v18q1"].fillna(0)

# v2a1: missing = no arrienda → imputar 0 + dummy
df["paga_arriendo"] = df["v2a1"].notna().astype(int)
df["v2a1"] = df["v2a1"].fillna(0)

# rez_esc: missing = no aplica por edad → imputar 0 + dummy
df["tiene_retraso"] = df["rez_esc"].notna().astype(int)
df["rez_esc"] = df["rez_esc"].fillna(0)

# meaneduc: 5 filas puntuales → imputar mediana
df["meaneduc"] = df["meaneduc"].fillna(df["meaneduc"].median())

print(f"\n[IMPUTACIONES] Missing restantes: {df.isnull().sum().sum()}")
print(f"  Shape tras imputaciones sección 2: {df.shape}")


[IMPUTACIONES] Missing restantes: 0
  Shape tras imputaciones sección 2: (7187, 129)


C:\Users\David\AppData\Local\Temp\ipykernel_9668\2169209631.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["paga_arriendo"] = df["v2a1"].notna().astype(int)
C:\Users\David\AppData\Local\Temp\ipykernel_9668\2169209631.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["tiene_retraso"] = df["rez_esc"].notna().astype(int)


In [6]:
# 3. Dummies con varianza cerca a cero

dummy_cols = [c for c in df.columns if df[c].nunique() == 2 and df[c].min() == 0]
low_var = []
for c in dummy_cols:
    pct_mode = df[c].value_counts(normalize=True).iloc[0] * 100
    if pct_mode >= 98:
        low_var.append({"variable": c, "pct_modo_dominante": round(pct_mode, 2)})

low_var_df = pd.DataFrame(low_var).sort_values("pct_modo_dominante", ascending=False)
print(f"\n{len(low_var_df)} variables con varianza casi cero:")
print(low_var_df.to_string(index=False))



32 variables con varianza casi cero:
      variable  pct_modo_dominante
       planpri               99.96
     pisoother               99.93
        noelec               99.89
     pisonatur               99.86
    paredother               99.85
     elimbasu6               99.83
     elimbasu4               99.83
 energcocinar1               99.83
     techootro               99.82
   paredfibras               99.81
   parentesco8               99.78
     techocane               99.75
    sanitario6               99.69
  parentesco10               99.67
   abastaguano               99.57
    sanitario1               99.54
          v14a               99.40
  parentesco12               99.18
   parentesco5               99.15
      pareddes               99.11
     paredzinc               99.03
   parentesco7               98.96
   parentesco4               98.78
  parentesco11               98.71
   parentesco9               98.62
    instlevel9               98.57
    instlevel7   

In [7]:
## Depuración


# 1. Definimos las que SÍ se eliminan (Baja varianza sin valor predictivo social claro)
vars_to_drop = [
    'planpri', 'pisoother', 'pisonatur', 'paredother', 'elimbasu6', 
    'energcocinar1', 'techootro', 'paredfibras', 'parentesco8', 
    'techocane', 'sanitario6', 'parentesco10', 'parentesco12', 'parentesco5', 
    'paredzinc', 'parentesco7', 'parentesco4', 'parentesco11', 
    'parentesco9', 'techoentrepiso'
]

# 2. Estas variables se PRESERVAN. Aunque tienen baja varianza, son indicadores críticos de carencia o bienestar.
preservar_criticas = [
    'noelec',          # Sin electricidad
    'abastaguano',     # Sin agua
    'sanitario1',      # Sin inodoro
    'sanitario5',      # Inodoro a letrina/pozo (Pobreza rural/precaria)
    'v14a',            # Tiene baño (Básico)
    'instlevel6',      # Técnica incompleta
    'instlevel7',      # Técnica completa
    'instlevel9',      # Postgrado (Indicador de no-pobreza/riqueza)
    'tipovivi4',       # Vivienda precaria (Tugurio/Asentamiento)
    'pisonotiene',     # Piso de tierra
    'pareddes'         # Paredes de material de desecho
]

# 3. Ejecución del drop asegurando que no eliminamos las de la lista de las que se quedan
# Filtramos vars_to_drop para que no contenga ninguna de preservar_criticas
final_drop = [v for v in vars_to_drop if v not in preservar_criticas]

df.drop(columns=[c for c in final_drop if c in df.columns], inplace=True)

print(f"Se eliminaron {len(final_drop)} variables con varianza casi cero.")
print(f"Se preservaron {len([v for v in preservar_criticas if v in df.columns])} variables críticas para la focalización:")
print(f"   {preservar_criticas}")
print(f"--- Nuevo shape: {df.shape} ---")

Se eliminaron 20 variables con varianza casi cero.
Se preservaron 11 variables críticas para la focalización:
   ['noelec', 'abastaguano', 'sanitario1', 'sanitario5', 'v14a', 'instlevel6', 'instlevel7', 'instlevel9', 'tipovivi4', 'pisonotiene', 'pareddes']
--- Nuevo shape: (7187, 109) ---


In [8]:
# 4. Outliers en variables continuas clave

cont_vars = ["age", "escolari", "meaneduc", "overcrowding", "dependency",
             "hogar_total", "rooms", "bedrooms", "v2a1"]

stats = df[cont_vars].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
print(stats[["min","1%","5%","50%","95%","99%","max"]].round(2).to_string())

              min    1%    5%    50%       95%       99%        max
age           0.0  1.00  4.00  31.00      72.0      85.0       97.0
escolari      0.0  0.00  0.00   6.00      15.0      17.0       21.0
meaneduc      0.0  0.95  3.00   8.67      16.0      21.0       37.0
overcrowding  0.2  0.50  0.67   1.50       3.0       5.0        6.0
hogar_total   1.0  1.00  2.00   4.00       7.0      10.0       12.0
rooms         1.0  1.00  3.00   5.00       8.0       9.0       11.0
bedrooms      1.0  1.00  1.00   3.00       4.0       5.0        8.0
v2a1          0.0  0.00  0.00   0.00  220000.0  482800.0  1000000.0


In [9]:
# Capping de v2a1 (Renta mensual) al percentil 99
# Evitamos que valores extremos (1,000,000) distorsionen la frontera de pobreza
p99_renta = df['v2a1'].quantile(0.99)
df['v2a1'] = df['v2a1'].clip(upper=p99_renta)

In [ ]:
# Examinamos las variables edjefe y edjefa (años educación jefe de hogar masculino/femenino)

print("\nDistribución de edjefe (años educación jefe masculino):")
print(pd.to_numeric(df["edjefe"], errors="coerce").value_counts().head(10))
print("\nDistribución de edjefa (años educación jefe femenino):")
print(pd.to_numeric(df["edjefa"], errors="coerce").value_counts().head(10))

both_zero = ((pd.to_numeric(df["edjefe"], errors="coerce").fillna(0) == 0) &
             (pd.to_numeric(df["edjefa"], errors="coerce").fillna(0) == 0)).sum()
print(f"\nHogares con edjefe=0 Y edjefa=0: {both_zero}")


Distribución de edjefe (años educación jefe masculino):
edjefe
6.0     1451
11.0     586
9.0      361
3.0      238
15.0     191
5.0      184
7.0      175
8.0      165
2.0      154
17.0     139
Name: count, dtype: int64

Distribución de edjefa (años educación jefe femenino):
edjefa
6.0     684
11.0    307
9.0     166
8.0     148
7.0     141
15.0    139
5.0     130
3.0     120
4.0     106
16.0     80
Name: count, dtype: int64

Hogares con edjefe=0 Y edjefa=0: 489


In [11]:
# 5. Estandarización variable edjefe y edjefa

def clean_edjefe(column):
    # Reemplazar "yes" por 1 y "no" por 0, luego convertir a float
    return pd.to_numeric(column.replace({'yes': 1, 'no': 0}), errors='coerce').fillna(0)

# Aplicar la limpieza a ambas columnas
df['edjefe'] = clean_edjefe(df['edjefe'])
df['edjefa'] = clean_edjefe(df['edjefa'])

# Creación de variable consolidada de educación del jefe de hogar (tomamos el valor máximo ya que solo uno debería ser el jefe)
df['ed_jefe_final'] = df[['edjefe', 'edjefa']].max(axis=1)

# Creación de variable de hogares sin cabeza de hogar
df['sin_jefe_educado'] = ((df['edjefe'] == 0) & (df['edjefa'] == 0)).astype(int)

print(f"Hay {df['sin_jefe_educado'].sum()} registros sin cabeza de hogar")

Hay 327 registros sin cabeza de hogar


C:\Users\David\AppData\Local\Temp\ipykernel_9668\2652289151.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ed_jefe_final'] = df[['edjefe', 'edjefa']].max(axis=1)
C:\Users\David\AppData\Local\Temp\ipykernel_9668\2652289151.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sin_jefe_educado'] = ((df['edjefe'] == 0) & (df['edjefa'] == 0)).astype(int)


In [12]:
#Reparación de hogares sin cabeza de hogar

# 1. Identificar hogares que tienen jefe
hogares_con_jefe = df[df['parentesco1'] == 1]['idhogar'].unique()

# 2. Identificar hogares que NO tienen jefe
todos_los_hogares = df['idhogar'].unique()
hogares_sin_jefe = set(todos_los_hogares) - set(hogares_con_jefe)

# 3. Corregir: Asignar como jefe a la persona con más educación en esos hogares
for hogar in hogares_sin_jefe:
    # Filtramos los miembros de ese hogar específico
    miembros = df[df['idhogar'] == hogar]
    
    # Encontramos el ID (índice) de la persona con más escolaridad
    # (Si hay empate, idxmax toma el primero, que suele ser el de más edad)
    idx_jefe_facto = miembros['escolari'].idxmax()
    
    # Le asignamos el rol de jefe de hogar
    df.at[idx_jefe_facto, 'parentesco1'] = 1

print(f"Se asignaron {len(hogares_sin_jefe)} jefes de hogar de facto basados en escolaridad.")

# 4. Ahora sí, recalculamos ed_jefe_final con la información corregida
df['ed_jefe_final'] = df[['edjefe', 'edjefa']].max(axis=1)

# Si el ed_jefe_final sigue siendo 0 pero la persona ahora marcada como jefe tiene escolaridad, actualizamos:
df.loc[(df['parentesco1'] == 1) & (df['ed_jefe_final'] == 0), 'ed_jefe_final'] = df['escolari']

Se asignaron 9 jefes de hogar de facto basados en escolaridad.


In [ ]:

# 5b. Reconstrucción de dependency desde variables demográficas del hogar
# Fórmula original (verificada contra valores numéricos del dataset, correlación = 1.0):
#   tasa_dependencia = (hogar_nin + hogar_mayor) / (hogar_adul - hogar_mayor)
# Donde:
#   - hogar_nin   : personas 0-19 años (dependientes jóvenes)
#   - hogar_mayor : personas 65+       (dependientes mayores)
#   - hogar_adul  : total adultos en el hogar (incluye 65+)
#   → denominador = adultos en edad de trabajar (18-64)
# Cuando el denominador es 0 (todos los adultos son mayores de 65),
# el dataset original usa 8 como valor techo (convención del INEC Costa Rica).

num = df['hogar_nin'] + df['hogar_mayor']
den = df['hogar_adul'] - df['hogar_mayor']

df['tasa_dependencia'] = np.where(den == 0, 8.0, num / den)

# Eliminamos la variable original contaminada
df.drop(columns=['dependency'], inplace=True)

print("=== Reconstrucción de tasa_dependencia ===")
print(df['tasa_dependencia'].describe().round(3))
print(f"\nHogares con tasa = 0  (sin dependientes):    {(df['tasa_dependencia'] == 0).sum()}")
print(f"Hogares con tasa = 8  (sin activos, techo):  {(df['tasa_dependencia'] == 8).sum()}")
print(f"Hogares con tasa > 1  (más dependientes que activos): {(df['tasa_dependencia'] > 1).sum()}")

# --- Distribución por nivel de pobreza ---
print("\n=== Tasa de dependencia media por nivel de Target ===")
print(df.groupby('Target')['tasa_dependencia'].mean().round(3).rename(
    {1: '1-Extrema', 2: '2-Moderada', 3: '3-Vulnerable', 4: '4-No pobre'}))

# --- Visualización ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Distribución general
axes[0].hist(df[df['tasa_dependencia'] < 8]['tasa_dependencia'], bins=25,
             color='steelblue', edgecolor='white')
axes[0].axvline(1, color='crimson', linestyle='--', label='Tasa = 1 (equilibrio)')
axes[0].set_title('Distribución tasa_dependencia\n(excluye techo=8)')
axes[0].set_xlabel('Tasa de dependencia')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# 2. Media por nivel de pobreza (Target original 1-4)
media_target = df.groupby('Target')['tasa_dependencia'].mean()
colores = ['#d73027', '#fc8d59', '#fee090', '#91bfdb']
bars = axes[1].bar([f'Target {i}' for i in media_target.index],
                    media_target.values, color=colores, edgecolor='white')
axes[1].set_title('Tasa de dependencia media\npor nivel de pobreza')
axes[1].set_ylabel('Media tasa_dependencia')
for bar, val in zip(bars, media_target.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# 3. Boxplot por Target
df.boxplot(column='tasa_dependencia', by='Target', ax=axes[2],
           notch=False, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('Dispersión por nivel de pobreza')
axes[2].set_xlabel('Target (1=extrema, 4=no pobre)')
axes[2].set_ylabel('Tasa de dependencia')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"\nShape tras reconstrucción: {df.shape}")


In [ ]:

# 6. Variables adicionales a eliminar

# Lista de variables a eliminar tras el análisis de relevancia y consistencia
vars_to_drop = [
    # 1. VARIANZA CASI CERO (Sin valor predictivo social claro)
    'planpri', 'pisoother', 'pisonatur', 'paredother', 'elimbasu6', 
    'elimbasu4', 'energcocinar1', 'techootro', 'paredfibras', 'techocane', 
    'sanitario6', 'paredzinc', 'techoentrepiso',

    # 2. REDUNDANCIA DE PARENTESCO
    # Se eliminan todos excepto parentesco1 (Jefe de hogar)
    'parentesco2', 'parentesco3', 'parentesco4', 'parentesco5', 'parentesco6', 
    'parentesco7', 'parentesco8', 'parentesco9', 'parentesco10', 'parentesco11', 
    'parentesco12',

    # 3. REDUNDANCIA DEMOGRÁFICA Y GÉNERO
    'r4h1', 'r4h2', 'r4h3', 'r4m1', 'r4m2', 'r4m3', 'r4t1', 'r4t2', 'r4t3',
    'male', # male es redundante con female

    # 4. EDUCACIÓN DESAGREGADA 
    # Mantenemos instlevel6, instlevel7 e instlevel9 por ser predictores clave de capital humano
    'instlevel1', 'instlevel2', 'instlevel3', 'instlevel4', 'instlevel5', 'instlevel8',

    # 5. ESTADO CIVIL
    'estadocivil1', 'estadocivil2', 'estadocivil3', 'estadocivil4', 
    'estadocivil5', 'estadocivil6', 'estadocivil7',

    # 6. INFRAESTRUCTURA ESPEJO (Multicolinealidad perfecta / Referencias)
    'area2',   # Capturado por area1
    'epared3', # Nivel 'bueno' (referencia)
    'etecho3', # Nivel 'bueno' (referencia)
    'eviv3',   # Nivel 'bueno' (referencia)

    # 7. dependency: eliminada en sección 5b y reemplazada por tasa_dependencia
    #    (reconstruida desde hogar_nin, hogar_mayor, hogar_adul con la fórmula original)
]

# --- NOTA DE SEGURIDAD ---
# Las siguientes variables han sido REMOVIDAS de vars_to_drop para ser PRESERVADAS:
# - 'pisonotiene', 'pareddes', 'tipovivi4', 'sanitario5' (Pobreza Extrema)
# - 'instlevel6', 'instlevel7', 'instlevel9' (Niveles educativos clave)
# - 'v14a', 'noelec', 'abastaguano', 'sanitario1' (Carencias críticas)

df.drop(columns=[c for c in vars_to_drop if c in df.columns], inplace=True)

print(f"Limpieza completada con éxito.")
print(f"Variables de carencia crítica y niveles de postgrado han sido protegidas.")
print(f"Dimensiones actuales: {df.shape}")


# Binarización de la Variable Target

In [14]:
# 1. Crear la variable binaria individual:
# 1 si Target es 1 o 2 (Pobre)
# 0 si Target es 3 o 4 (No Pobre)
df['Target_binario'] = df['Target'].apply(lambda x: 1 if x <= 2 else 0)

# 2. Aplicar la Regla de la Mayoría por Hogar
# Creamos un diccionario donde la llave es el idhogar y el valor es la moda del Target_binario
target_hogar = df.groupby('idhogar')['Target_binario'].agg(
    lambda x: x.mode().max() # .max() asegura que en empate 0 y 1, elija 1 (Pobre)
).to_dict()

# 3. Mapear el nuevo Target consistente a todos los miembros del hogar
df['Target_final'] = df['idhogar'].map(target_hogar)

# Verificación de la distribución
print("Distribución del Target Original (Individuos):")
print(df['Target'].value_counts(normalize=True).round(4) * 100)

print("\nDistribución del Target Binario Final (Consistente por Hogar):")
print(df['Target_final'].value_counts(normalize=True).round(4) * 100)

# Eliminar las columnas intermedias para mantener la limpieza
df.drop(columns=['Target', 'Target_binario'], inplace=True)
df.rename(columns={'Target_final': 'Target'}, inplace=True)

Distribución del Target Original (Individuos):
Target
4    61.78
2    17.23
3    12.80
1     8.20
Name: proportion, dtype: float64

Distribución del Target Binario Final (Consistente por Hogar):
Target_final
0    74.66
1    25.34
Name: proportion, dtype: float64


# Agregación por hogar con idhogar

In [15]:
# 1. Definir las reglas de agregación
agg_rules = {}

for col in df.columns:
    if col in ['idhogar', 'Target']:
        continue
    # Variables de infraestructura, activos y jefe: tomamos el máximo
    elif col in ['v2a1', 'rooms', 'bedrooms', 'overcrowding', 'ed_jefe_final', 
                 'sin_jefe_educado', 'paga_arriendo', 'refrig', 'computer', 'television', 'rez_esc']:
        agg_rules[col] = 'max'
    # Variables que son promedios o proporciones: tomamos la media
    elif col in ['escolari', 'meaneduc', 'age', 'hogar_nin', 'hogar_adul', 'hogar_mayor', 'hogar_total', 'dependency']:
        agg_rules[col] = 'mean'
    # Para el resto de dummies (infraestructura, ubicación, etc.): tomamos el máximo
    else:
        agg_rules[col] = 'max'

# 2. Ejecutar el GroupBy por idhogar
df_hogar = df.groupby('idhogar').agg({**agg_rules, 'Target': 'first'}).reset_index()

# 3. Verificación final
print(f"Dataset original (individuos): {df.shape}")
print(f"Dataset final (hogares): {df_hogar.shape}")

display(df_hogar.head())

# Guardar el dataset limpio para el siguiente punto del taller
df_hogar.to_csv("train_cleaned_hogar.csv", index=False)

Dataset original (individuos): (7187, 83)
Dataset final (hogares): (2241, 83)


,idhogar,Id,v2a1,hacdor,rooms,hacapo,v14a,refrig,v18q,v18q1,escolari,rez_esc,paredblolad,paredzocalo,paredpreb,pareddes,paredmad,pisomoscer,pisocemento,pisonotiene,pisomadera,techozinc,cielorazo,abastaguadentro,abastaguafuera,...,instlevel9,bedrooms,overcrowding,tipovivi1,tipovivi2,tipovivi3,tipovivi4,tipovivi5,computer,television,mobilephone,qmobilephone,lugar1,lugar2,lugar3,lugar4,lugar5,lugar6,area1,age,paga_arriendo,tiene_retraso,ed_jefe_final,sin_jefe_educado,Target
0,001ff74ca,ID_654683e33,0.0,0,6,0,1,1,1,1.0,8.00,0.0,1,0,0,0,0,1,0,0,0,0,1,1,0,...,0,4,0.500000,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,19.00,0,0,16,0,0
1,003123ec2,ID_99c9bcea5,0.0,0,3,0,1,1,0,0.0,3.25,0.0,0,0,1,0,0,0,1,0,0,1,0,1,0,...,0,2,2.000000,0,0,0,0,1,0,0,1,2,0,0,0,0,1,0,1,12.75,0,0,6,0,1
2,004616164,ID_fe8c32eba,0.0,0,4,0,1,1,0,0.0,7.00,0.0,0,0,0,0,1,0,0,0,1,1,0,0,1,...,0,3,0.666667,0,0,0,0,1,0,0,1,2,0,1,0,0,0,0,0,33.00,0,1,3,0,1
3,004983866,ID_5ad4372cd,0.0,0,5,0,1,1,0,0.0,7.50,2.0,0,0,0,0,1,0,1,0,0,1,0,1,0,...,0,2,1.000000,1,0,0,0,0,0,0,1,2,0,0,1,0,0,0,1,37.50,0,1,8,0,0
4,006b64543,ID_d2317b09a,0.0,0,4,0,1,1,0,0.0,7.00,0.0,1,0,0,0,0,1,0,0,0,1,1,1,0,...,0,2,1.000000,1,0,0,0,0,0,0,1,2,0,0,0,1,0,0,1,43.00,0,0,3,0,0


# Análisis de Multicolinealidad Perfecta con VIF

In [16]:
# Analizamos multicolinealidad perfecta

from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Preparar matriz X (solo numéricas, sin Target ni ID)
X_vif = df_hogar.select_dtypes(include=[np.number]).drop(columns=['Target'], errors='ignore')

# 2. Calcular VIF
vif_data = pd.DataFrame()
vif_data["variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

# 3. Filtrar resultados preocupantes
print("--- Variables con VIF > 10 (Alta redundancia) ---")
print(vif_data[vif_data["VIF"] > 10].sort_values("VIF", ascending=False).head(15))

# 4. Buscar Infinitos
if np.isinf(vif_data["VIF"]).any():
    print("\n Variables con correlación perfecta (infinito).")
    print(vif_data[np.isinf(vif_data["VIF"])]["variable"].tolist())

c:\Users\David\Desktop\Universidad\Semestre 2026-10\Consultoría Económica con IA\Repositorios\Sandbox_HE2_DavidRodriguez\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
c:\Users\David\Desktop\Universidad\Semestre 2026-10\Consultoría Económica con IA\Repositorios\Sandbox_HE2_DavidRodriguez\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
c:\Users\David\Desktop\Universidad\Semestre 2026-10\Consultoría Económica con IA\Repositorios\Sandbox_HE2_DavidRodriguez\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


--- Variables con VIF > 10 (Alta redundancia) ---
           variable  VIF
48       hogar_adul  inf
47        hogar_nin  inf
23      abastaguano  inf
22   abastaguafuera  inf
21  abastaguadentro  inf
72           lugar5  inf
73           lugar6  inf
76    paga_arriendo  inf
71           lugar4  inf
78    ed_jefe_final  inf
70           lugar3  inf
69           lugar2  inf
68           lugar1  inf
63        tipovivi5  inf
62        tipovivi4  inf

 Variables con correlación perfecta (infinito).
['abastaguadentro', 'abastaguafuera', 'abastaguano', 'hogar_nin', 'hogar_adul', 'hogar_total', 'edjefe', 'edjefa', 'tipovivi1', 'tipovivi2', 'tipovivi3', 'tipovivi4', 'tipovivi5', 'lugar1', 'lugar2', 'lugar3', 'lugar4', 'lugar5', 'lugar6', 'paga_arriendo', 'ed_jefe_final']


In [17]:
# 1. Variables a eliminar justificadas por grupos lógicos:
cols_to_drop_final = [
    # --- REDUNDANCIAS POR COMPOSICIÓN DEL HOGAR ---
    'hogar_total',      # Es la suma exacta de hogar_nin + hogar_adul.
    'parentesco1',      # En el dataset de hogares, casi siempre es 1 (constante).
    
    # --- REDUNDANCIAS DE CAPITAL HUMANO ---
    'edjefe', 'edjefa', # Ya consolidadas en nuestra variable 'ed_jefe_final'.
    
    # --- TRAMPA DE VARIABLES FICTICIAS (DUMMIES) ---
    # Eliminamos una de cada grupo para que sirva como categoría de referencia
    'lugar6',           # Referencia para región geográfica.
    'abastaguadentro',  # Referencia para acceso a agua.
    'tipovivi1',        # Referencia para tenencia de vivienda.
    'tipovivi3',        # Eliminada adicionalmente para romper el ciclo 'inf' en vivienda.
    
    # --- REDUNDANCIAS DE ACTIVOS Y GASTOS ---
    'paga_arriendo',    # Redundancia perfecta con 'v2a1' (si v2a1 > 0, paga arriendo).
    'v18q',             # Redundante con 'v18q1' (indica si tiene tablet).
    'public',           # Redundante con los indicadores específicos de electricidad.
    
    # --- VARIABLES DE ALTA COLINEALIDAD (VIF > 50) ---
    # Simplificamos para mejorar la estabilidad de la Regresión Logística
    'energcocinar2',    # Gas de cilindro (muy correlacionada con nivel socioeconómico).
    'energcocinar3',     # Gas de red/electricidad.

    # --- NO ÚTILES ---
    'Id' # Ya no lo necesitamos.
]

# 2. Ejecutamos la eliminación en el dataset de hogares
df_model_final = df_hogar.drop(columns=[c for c in cols_to_drop_final if c in df_hogar.columns])

# 3. Verificación de seguridad con VIF (Calculado sobre el dataset final)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Seleccionamos predictores numéricos (X) y el objetivo (y)
X_final = df_model_final.select_dtypes(include=[np.number]).drop(columns=['Target'], errors='ignore')
y_final = df_model_final['Target']

vif_report = pd.DataFrame()
vif_report["variable"] = X_final.columns
vif_report["VIF"] = [variance_inflation_factor(X_final.values, i) for i in range(len(X_final.columns))]

# 4. Resultados finales
print(f"Variables restantes: {len(df_model_final.columns)}")
print(f"Infinitos detectados: {np.isinf(vif_report['VIF']).sum()}")
print("\nTop 5 variables con mayor VIF:")
print(vif_report.sort_values("VIF", ascending=False).head(5))

c:\Users\David\Desktop\Universidad\Semestre 2026-10\Consultoría Económica con IA\Repositorios\Sandbox_HE2_DavidRodriguez\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


Variables restantes: 69
Infinitos detectados: 0

Top 5 variables con mayor VIF:
       variable         VIF
14   pisomoscer  336.395625
4          v14a  256.173358
26   sanitario3  226.121785
29    elimbasu1  214.541139
15  pisocemento  108.347699


# Renombramos variables a nombres descriptivos

In [ ]:

# Diccionario de traducción de códigos a nombres descriptivos

diccionario_nombres = {
    # Vivienda Básicos y Activos
    'v2a1': 'Monto_Alquiler',
    'hacdor': 'Hacinamiento_Dormitorios',
    'rooms': 'Total_Habitaciones',
    'hacapo': 'Hacinamiento_Cuartos',
    'v14a': 'Tiene_Bano',
    'refrig': 'Tiene_Nevera',
    'v18q1': 'Cantidad_Tablets',
    'computer': 'Tiene_Computador',
    'television': 'Tiene_TV',
    'mobilephone': 'Tiene_Celular',
    'qmobilephone': 'Cant_Celulares',
    
    # Educación (General e Interacciones)
    'escolari': 'Promedio_Anos_Escolaridad',
    'rez_esc': 'Rezago_Escolar',
    'meaneduc': 'Promedio_Educ_Adultos',
    'ed_jefe_final': 'Educacion_Jefe_Hogar',
    'sin_jefe_educado': 'Hogar_sin_Jefe_Educado',
    'tiene_retraso': 'Hogar_con_Rezago_Escolar',
    
    # Educación Niveles Específicos (Protegidos)
    'instlevel6': 'Educ_Tecnica_Incompleta',
    'instlevel7': 'Educ_Tecnica_Completa',
    'instlevel9': 'Educ_Postgrado',

    # Materiales de Construcción (Paredes, Piso, Techo)
    'paredblolad': 'Pared_Bloque_Ladrillo',
    'paredzocalo': 'Pared_Zocalo',
    'paredpreb': 'Pared_Prefabricado',
    'pareddes': 'Pared_Desecho',
    'paredmad': 'Pared_Madera',
    'pisomoscer': 'Piso_Mosaico_Ceramica',
    'pisocemento': 'Piso_Cemento',
    'pisonotiene': 'Piso_Tierra',
    'pisomadera': 'Piso_Madera',
    'techozinc': 'Techo_Zinc',
    'cielorazo': 'Tiene_Cielorazo',
    
    # Estado de la Infraestructura
    'epared1': 'Pared_Mala', 'epared2': 'Pared_Regular',
    'etecho1': 'Techo_Malo', 'etecho2': 'Techo_Regular',
    'eviv1': 'Piso_Malo', 'eviv2': 'Piso_Regular',
    
    # Servicios Básicos
    'abastaguafuera': 'Agua_Fuera_Vivienda',
    'abastaguano': 'Sin_Abasto_Agua',
    'noelec': 'Sin_Electricidad',
    'coopele': 'Electricidad_Cooperativa',
    'sanitario1': 'Sin_Inodoro',
    'sanitario2': 'Inodoro_Alcantarillado',
    'sanitario3': 'Inodoro_Septico',
    'sanitario5': 'Inodoro_Letrina',
    'energcocinar4': 'Cocina_Carbon_Lena',
    'elimbasu1': 'Basura_Camion',
    'elimbasu2': 'Basura_Enterrada',
    'elimbasu3': 'Basura_Quemada',
    'elimbasu5': 'Basura_Rio_Creek',
    
    # Demografía
    'dis': 'Persona_Discapacitada',
    'female': 'Proporcion_Mujeres',
    'hogar_nin': 'Cant_Ninos',
    'hogar_adul': 'Cant_Adultos',
    'hogar_mayor': 'Cant_Adultos_Mayores',
    'age': 'Edad_Promedio',
    'bedrooms': 'Total_Dormitorios',
    'overcrowding': 'Personas_por_Cuarto',
    'tasa_dependencia': 'Tasa_Dependencia',

    # Régimen de Tenencia y Ubicación
    'tipovivi2': 'Casa_Propia_Pagando',
    'tipovivi4': 'Casa_Precario',
    'tipovivi5': 'Casa_Prestada_Asignada',
    'lugar1': 'Region_Central', 'lugar2': 'Region_Chorotega',
    'lugar3': 'Region_Pacifico_Central', 'lugar4': 'Region_Brunca',
    'lugar5': 'Region_Huetar_Atlantica',
    'area1': 'Zona_Urbana'
}

# Aplicar el cambio de nombre
df_model_final.rename(columns=diccionario_nombres, inplace=True)

print(f"Variables renombradas.")
print(f"Total de columnas: {len(df_model_final.columns)}")
print("-" * 30)
print(df_model_final.columns.tolist())


In [ ]:
# Eliminación final de variables que no aportan post-agregación

# Lista consolidada de variables a eliminar para mejorar la estabilidad del modelo
final_drop_list = [
    # --- 1. REDUNDANCIAS DE AGREGACIÓN (Nivel Individuo vs Hogar) ---
    'Tiene_Bano',               # Redundante con tipos de inodoro específicos (VIF alto).
    'Tiene_Celular',            # Varianza casi cero; 'Cant_Celulares' es más informativo.
    'Total_Habitaciones',       # Se prefiere 'Total_Dormitorios' para medir activos físicos.
    'Anos_Escolaridad_Prom',    # Se prefiere 'Promedio_Educ_Adultos' por estabilidad.
    'Rezago_Escolar',           # Se prefiere la binaria 'Hogar_con_Rezago_Escolar'.

    # --- 2. CLÚSTER DE HACINAMIENTO (Dejamos Personas_por_Cuarto y Total_Dormitorios) ---
    'Hacinamiento_Dormitorios', 
    'Hacinamiento_Cuartos',

    # --- 3. CLÚSTER DE EDUCACIÓN (Dejamos Promedio_Educ_Adultos y la Binaria de Vulnerabilidad) ---
    'Educacion_Jefe_Hogar',     # Los años del jefe suelen colinear con el promedio de adultos.

    # --- 4. CLÚSTER DE MATERIALES (Dejamos el Estado General: Mala/Regular) ---
    'Pared_Desecho',            # Suele estar implícito en 'Pared_Mala'.
    'Piso_Tierra'               # Suele estar implícito en 'Piso_Malo'.
]

# Ejecución de la eliminación
df_model_final.drop(columns=[c for c in final_drop_list if c in df_model_final.columns], inplace=True)

# Verificación de resultados
print(f"Variables eliminadas: {len(final_drop_list)}")
print(f"--- Dimensiones: {df_model_final.shape} ---")

Variables eliminadas: 10
--- Dimensiones definitivas para entrenamiento: (2241, 60) ---
